# Is your backtest real? Check it in 30 seconds

[TokIO](https://github.com/jordanahern2009-svg/Tokio-ai) asks whether a backtest's profit is distinguishable from luck, once two things are allowed for:

1. **Serial dependence.** Positions held for weeks make daily P&L far from independent; a plain t-test counts every bar as fresh evidence.
2. **Your search.** The Sharpe on your screen is the best of every variant you tried. The best of twenty zero-edge strategies looks good by construction.

Run the cells top to bottom. No API key, no account.

In [ ]:
%pip install -q "tokio-ai>=0.5" yfinance

## 1. A grid everyone has run: SMA crossovers on SPY

Nineteen fast/slow pairs, long when fast > slow and short otherwise, on 20 years of daily SPY. The position is decided at the close and held over the next bar, so there is no lookahead.

In [ ]:
import pandas as pd
import yfinance as yf
import tokio_ai

close = yf.download("SPY", period="20y", auto_adjust=True, progress=False)["Close"].squeeze()
r = close.pct_change()

pnl, positions = {}, {}
for fast in (5, 10, 20, 50):
    for slow in (50, 100, 150, 200, 250):
        if fast >= slow:
            continue
        f, s = close.rolling(fast).mean(), close.rolling(slow).mean()
        pos = (f > s).astype(float) * 2 - 1          # +1 long, -1 short
        pos = pos.where(s.notna()).shift(1)           # known at the close, held over the next bar
        name = f"sma{fast}/{slow}"
        pnl[name], positions[name] = pos * r, pos
print(len(pnl), "variants")

## 2. The best one, the way it usually gets reported

In [ ]:
best = max(pnl, key=lambda k: pnl[k].mean() / pnl[k].std())
print(best)
print(tokio_ai.check_backtest(pnl[best], positions=positions[best]))

## 3. The same backtest, honestly: every variant you tried

Pass the whole grid. The verdict is about the best variant, corrected for all of them using how correlated they are, plus the probability of backtest overfitting (does the in-sample winner keep winning out of sample?).

In [ ]:
print(tokio_ai.check_backtest(pnl, positions=positions))

## 4. With trading costs

`costs=0.0005` charges 5 bps per unit of position traded. The output also gives the **breakeven cost**: the most you could pay and still have a significant result.

In [ ]:
print(tokio_ai.check_backtest(pnl, positions=positions, costs=0.0005))

## 5. Your own backtest

**From vectorbt or backtesting.py**, pass the object itself; returns and positions are read from it:

```python
pf = vbt.Portfolio.from_signals(close, entries, exits, fees=0.001)   # a whole grid works too
print(tokio_ai.check_backtest(pf))

stats = Backtest(data, MyStrategy, commission=0.001).run()
print(tokio_ai.check_backtest(stats))
```

**From anything else**, upload a CSV with one column of per-bar returns per variant (a date column is skipped), then run the cell below. If you tried more variants than you're uploading, set `TRIALS` to the real number.

In [ ]:
TRIALS = None   # e.g. 40 if you tried 40 things and kept these

try:
    from google.colab import files
    name = next(iter(files.upload()))
except ImportError:
    name = input("path to your CSV: ")
df = pd.read_csv(name)
df = df.select_dtypes("number")
print(tokio_ai.check_backtest(df if df.shape[1] > 1 else df.iloc[:, 0], trials=TRIALS))

Found something odd? [Open an issue](https://github.com/jordanahern2009-svg/Tokio-ai/issues/new/choose); there's a template for statistical concerns. The false-positive rates behind every number here are measured and reproducible: [docs/calibration.md](https://github.com/jordanahern2009-svg/Tokio-ai/blob/main/docs/calibration.md).